# 03 - Clean PDF NER

Pipeline: PDF -> PyMuPDF -> GLiNER -> predicted entities -> strict/relaxed evaluation.

In [ ]:
import importlib
import ner.evaluation, ner.inference, ner.pdf, ner.utils

# reload so edits in ner/ are picked up without restarting the kernel
for module in (ner.utils, ner.pdf, ner.inference, ner.evaluation):
    importlib.reload(module)

from ner.evaluation import evaluate_ner_relaxed, evaluate_ner_strict
from ner.inference import load_gliner_model, predict_dataset
from ner.pdf import extract_pdf_dataset
from ner.utils import load_jsonl, save_responses_to_jsonl


In [2]:
VALIDATION_PATH = '../data/plain/validation.jsonl'
PDF_DIR = '../data/pdf_clean'
OUTPUT_PATH = '../results/pdf_clean/gliner_small_validation.jsonl'

LABELS = [
    'PERSON_NAME', 'ID_CARD_NUMBER', 'ADDRESS',
    'REGISTRATION_NUMBER', 'DATE', 'AMOUNT', 'ORGANIZATION'
]


In [3]:
gold_samples = load_jsonl(VALIDATION_PATH)
# Clean PDFs are named <id>.pdf
pdf_samples = extract_pdf_dataset(gold_samples, PDF_DIR, suffix='')
print(len(pdf_samples), 'PDFs;', sum(1 for s in pdf_samples if s['text'].strip()), 'with extractable text')
model = load_gliner_model()
responses = predict_dataset(model, pdf_samples, LABELS, threshold=0.5)
save_responses_to_jsonl(responses, OUTPUT_PATH)


50 PDFs; 50 with extractable text


C:\Users\paudel\PycharmProjects\ProjecteName\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

In [4]:
strict_results = evaluate_ner_strict(VALIDATION_PATH, OUTPUT_PATH)
relaxed_results = evaluate_ner_relaxed(VALIDATION_PATH, OUTPUT_PATH)



STRICT EVALUATION
Micro -> P: 0.52 | R: 0.59 | F1: 0.56 | TP: 1215 | FP: 1103 | FN: 835

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 0.62   0.62   0.62     308     192     192
ID_CARD_NUMBER              0.81   0.88   0.84     175      42      25
ADDRESS                     0.00   0.01   0.00       1     415     199
REGISTRATION_NUMBER         0.70   0.66   0.68     165      72      85
DATE                        0.65   0.65   0.65     261     138     139
AMOUNT                      0.79   0.76   0.77     152      41      48
ORGANIZATION                0.43   0.51   0.47     153     203     147

RELAXED EVALUATION - OCCURRENCES
Micro -> P: 0.76 | R: 0.86 | F1: 0.81 | TP: 1773 | FP: 545 | FN: 277

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------